# Local RAG pipeline (documented)

**What RAG is, in one line:** you turn your documents into numbers (embeddings), store them, then when a question comes in you turn the question into numbers too, find the closest document chunks, and hand those chunks to an LLM as context so it answers from *your* data instead of guessing.

Four stages, one cell each:

1. **Load** files from `doc/`
2. **Split** them into chunks and **embed** each chunk into a vector store (Chroma)
3. **Retrieve** the chunks nearest to a query
4. **Generate** an answer with an LLM using those chunks

## Setup (run once in a terminal, not here)

```bash
brew install uv
uv venv --python 3.12 && source .venv/bin/activate
uv pip install langchain langchain-core langchain-community langchain-text-splitters \
               langchain-chroma langchain-ollama langchain-google-genai pypdf python-dotenv ipykernel
ollama pull nomic-embed-text
```

Then select `.venv` as the kernel for this notebook (top right in VS Code).

Why 3.12 and not the newest Python: the compiled packages Chroma and the embedding stack depend on ship wheels for new Python versions months late. On the newest interpreter pip compiles from source, which is slow and fails often.

Why Ollama for embeddings: `HuggingFaceEmbeddings` requires `sentence-transformers`, which requires `torch` (2 GB, slow install, version-sensitive). Ollama runs the embedding model as a local server, so Python only needs an HTTP client.

In [4]:
import os, glob
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model

load_dotenv()  # reads GOOGLE_API_KEY from a .env file next to this notebook
assert os.getenv("GOOGLE_API_KEY"), "put GOOGLE_API_KEY=... in .env"

## 1. Load

A LangChain **Document** is just `page_content` (the text) plus `metadata` (a dict, usually the source path). Loaders turn files into Documents. Here we handle `.md`, `.txt` and `.pdf`; add more loaders as needed.

The `source` metadata matters later: it lets you show the user which file an answer came from.

In [5]:
DOC_DIR = "docs"

def load_docs(folder):
    docs = []
    for path in glob.glob(f"{folder}/**/*", recursive=True):
        if path.endswith(".pdf"):
            docs += PyPDFLoader(path).load()          # one Document per page
        elif path.endswith((".md", ".txt")):
            docs += TextLoader(path, encoding="utf-8").load()
    return docs

raw_docs = load_docs(DOC_DIR)
print(len(raw_docs), "documents loaded")
for d in raw_docs[:3]:
    print(" ", d.metadata["source"], len(d.page_content), "chars")

1 documents loaded
  docs/data.md 833 chars


## 2. Split and embed

**Why split:** embedding models have a token limit, and a whole file as one vector is too blurry to match a specific question. Chunks of a few hundred characters each get their own vector.

`RecursiveCharacterTextSplitter` tries to cut on paragraphs first, then lines, then sentences, then words, so chunks stay coherent. `chunk_overlap` repeats the tail of one chunk at the head of the next so a sentence split across the boundary still lands whole in at least one chunk.

**Embedding** maps text to a fixed-length vector (768 numbers for `nomic-embed-text`). Semantically similar text ends up geometrically close.

**Chroma** stores those vectors on disk and does the nearest-neighbour search. `persist_directory` means you index once and reuse across kernel restarts. Delete the `chroma_db` folder to rebuild from scratch.

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(raw_docs)   # keeps metadata on each chunk
print(len(chunks), "chunks")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

store = Chroma(
    collection_name="docs",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

# only embed if the store is empty, so re-running the notebook is cheap
if store._collection.count() == 0:
    store.add_documents(chunks)
    print("embedded and stored")
else:
    print("store already populated with", store._collection.count(), "chunks")

2 chunks
embedded and stored


## 3. Retrieve

The query goes through the **same** embedding model as the chunks (this is non-negotiable; different models produce incompatible vector spaces). Chroma then returns the `k` chunks with the smallest distance.

The score Chroma returns is a **distance**, so lower means more similar. Print it while learning; you'll quickly get a feel for what a good vs bad match looks like for your data.

In [7]:
q = "which method takes the longest"   # user input

hits = store.similarity_search_with_score(q, k=4)
for doc, score in hits:
    print(f"[{score:.3f}] {doc.metadata['source']}")
    print("   ", doc.page_content[:200].replace(chr(10), " "), "\n")

[0.967] docs/data.md
    # Coffee brewing  ## Espresso Espresso uses about 9 bars of pressure and a fine grind. A double shot pulls in 25 to 30 seconds from 18g of coffee, yielding roughly 36g in the cup. Grind too coarse and 

[1.142] docs/data.md
    ## Storage Whole beans keep for about a month after roast in an airtight container away from light. Ground coffee goes stale within days. Never refrigerate, the moisture ruins it. """ 



## 4. Generate

Stuff the retrieved chunks into the prompt and ask the model to answer from them only. That instruction is what keeps the LLM from falling back on its own training data when your docs don't cover the question.

`init_chat_model` is LangChain's provider-agnostic constructor. Swap the two arguments to switch backends:

| Backend | Call |
|---|---|
| Gemini | `init_chat_model("gemini-2.5-flash", model_provider="google_genai")` |
| Ollama local | `init_chat_model("llama3.2", model_provider="ollama")` |

Check the current Gemini model name in Google's docs; they rotate frequently.

In [9]:
llm = init_chat_model("gemini-3.6-flash", model_provider="google_genai")

ctx = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d, _ in hits)
prompt = f"""Answer using only the context below. If the context does not contain the answer, say so.

Context:
{ctx}

Question: {q}"""

print(llm.invoke(prompt).content)

[{'type': 'text', 'text': 'Based on the provided context, the French press method takes the longest, requiring a four-minute steep (compared to around three minutes for pour over and 25 to 30 seconds for espresso).', 'extras': {'signature': 'EuQICuEIARFNMg+d6zvakA/XY/4SO6cXxKAFR7j1KeMoMmkEU4rw2EnYCsl6tjpTE+kVKSQ1JLiVNDAsQYTuexqk8AqHFelq5KbkvO9bkUj0L+UANzF4Gs1teZDKYgjGC321RutgG0lDgMMyZmq7ljky2SPVsqhIMedfyJqNEv1dyugJXTjjEFDtfldXAD9xRgrWGZGk1wk103LJeTLZq5f1o7L+jDcRjwPZnJ+3UlIA18oFc18eUXtPUHzT6LJNSpolbnqWxlDvOY2wHQfOYYrB10V7o19PDXlqzpm/8SSDi0mLxxN+wve0CMisY08oIhkPeyzbDP9wFz2JgOMyft7o346iDAvDElpCH6VhMQoSSNpRm63AtT+mHCpHaoVeFiVB1tpODfWv0B+/CMe24dJePjRJ9eTjLTdevfWd4vqFaGpNwjUKubXqUkRF31uNDJHwA/U8J8AakIEa080+Zllu/E7frkps/EI6780Sm1hv+h+YObTA00rXGzNwmKNqG6acHL2vbk9KXAOmPA0acB3LQfj2C8i1wL4frGLwZOMGO85Ak2ae3Xf6swVlQxt/mylzv6+pg7ouE/qFG03szAsr+AXgMHeBnw3pcu/DjOBeBfc9435WUixn9/FbcjIBOJk1HwG8iR6yyPnE6Uq01DCXU8HvE+EuO7sPnCFs/NtlxbetUz86MehZAqVoa21yaSBaT+3HFHSv1+C2zTxRscHjjHO44f/9YJ00/5d4gk9u7W9dPGkFbO

## Wrap it in a function

Same four stages, callable. Everything above the retrieval step (loading, splitting, embedding) runs once; this runs per question.

In [10]:
def ask(question, k=4):
    hits = store.similarity_search(question, k=k)
    ctx = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in hits)
    prompt = f"Answer using only this context. If it is not in the context, say so.\n\n{ctx}\n\nQuestion: {question}"
    return llm.invoke(prompt).content

print(ask("how long should espresso take to pull?"))

[{'type': 'text', 'text': 'Based on the provided context, a double shot of espresso takes 25 to 30 seconds to pull.', 'extras': {'signature': 'Er0ICroIARFNMg9I40TULBVb0RWL7tYFtscPMvQQCcp9vnvD1vlC9TLbo4OGlataddBFxw21MMyxsgtlMuTFstLH0wiuhLzolSsogaDsDQBoprIorbtcZcMGHA81DwzZ4Tm3zGBR/RHIMVKPHibQsbgCBUSIeVFv2krsB4yGFI/bRQNBu/KOp9midAk7y4jvrrnKG9cubkHRalqQMT0jUpXS90nubtkRXTPNd8QQ+dUwN0Fk99sJgFElPBLdOHUgjbBv7av9KgNU3Twhc7cuTZ124AapubbbIfXLjZlIDXnRlT08DBv+TZ0aHpdIWfQf/4R7CLc5Ys95DDCFWstWn4CRq1l7Gmz5HzEtTt4kEadh6s6VL34QX7rts5O0aCtpShBbqnJFbRPb89vHeX9xhoehm1QXfgHfs7Ok/iaCuXLhWCtW588FXFDqUJV5pEV3OBkATl1Tvon2xKD68Lu3p6SbA3bx6V/4faJ6pqoxbWX+OqdSJ4P0eyJZbvX4yaCc7uA/9Adm0qEhrYVGFYpThuocrOkNwky9eg/PWd9V30HYEZdIE4awLy+x2LT6ewkH0Bo1DJEtMYKYmrf/MsNEMmt+iIbrHGVRwkX/ETskkhuEbc5NlfC9piehNPNhFqKki3e3eGMQmDWDiZquUXh0qRYEfRPvm4bIf0Ax+s+KIa+OoStUaL2lOIeufkm20zRO3MWjlb2HlMpsacoxqN+4hmcIezsZElaoXmt3zL6u911AxtjlJ46YllF5/PTjt05VtA7jVnh6wsP8FNQYtSWQ2wFYE9Ilzj22rhvGf+IGj/qdkA2XA2UcQk9QiU00axY24VrkGGRf9S20OfeNTGWok3LdU

## Where to go next

- **Chunk size tuning.** 800/120 is a starting point. Smaller chunks give precise matches but less context per hit; larger the reverse.
- **Hybrid search.** Vector search misses exact keywords (part numbers, names). Add BM25 (`langchain_community.retrievers.BM25Retriever`) and merge results.
- **Reranking.** Retrieve 20, rerank to 4 with a cross-encoder. Big quality jump for small cost.
- **Metadata filters.** `store.similarity_search(q, filter={"source": "doc/foo.pdf"})` restricts the search.
- **Swap the embedder back to HuggingFace** once your environment is stable: `langchain_huggingface.HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")`. Rebuild the Chroma store when you do, since vectors from different models don't mix.